# Advanced Multi-Model Training: Smart House Price Predictor

Trains 4 models on synthetic housing data, compares performance, generates SHAP feature importance, and exports the best model.

**Models:** Linear Regression, Random Forest, XGBoost, MLPRegressor (Neural Net)
**Selection criteria:** Lowest RMSE on test set

In [ ]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

print('All imports successful!')

## 1. Load Dataset
Synthetic dataset representing housing market (1000 houses, 8 features).

In [ ]:
df = pd.read_csv('house_data.csv')
print(f'Dataset Shape: {df.shape}')
print(f'Dataset Size: {len(df)} houses')
print(f'Features: {list(df.columns)}')
df.head()

In [ ]:
# Feature engineering (matching the original backend expectations)
categorical_cols = ['has_pool', 'has_garage', 'has_ac']

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Engineered features (used by the trained model)
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['bath_bed_ratio'] = df['bathrooms'] / (df['bedrooms'] + 1)

print('Encoded categorical columns')
print(f'Feature columns: {list(df.columns)}')

In [ ]:
# Split features and target
X = df.drop(['price'], axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train shape: {X_train.shape}')
print(f'Test shape: {X_test.shape}')
print(f'Price range: ₹{y.min():,.0f} — ₹{y.max():,.0f}')

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Scaler mean shape: {scaler.mean_.shape}')
print('Scaling complete')

## 2. Train 4 Models
Comparing Linear Regression, Random Forest, XGBoost, and MLPRegressor (Neural Net).

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=42
    ),
    'XGBoost': xgb.XGBRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, verbosity=0
    ),
    'MLPRegressor (Neural Net)': MLPRegressor(
        hidden_layer_sizes=(64, 32), activation='relu',
        solver='adam', max_iter=500, random_state=42,
        early_stopping=True, validation_fraction=0.1
    )
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)

    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)

    results[name] = {
        'R2': round(r2, 4),
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2)
    }
    trained_models[name] = model

    print(f'  R²: {r2:.4f}  RMSE: ₹{rmse:,.2f}  MAE: ₹{mae:,.2f}')

print('\nAll models trained!')

## 3. Model Comparison Table

In [ ]:
comparison_df = pd.DataFrame(results).T
comparison_df['Rank'] = comparison_df['RMSE'].rank().astype(int)
comparison_df = comparison_df.sort_values('RMSE')
comparison_df

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['R2', 'RMSE', 'MAE']
colors = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db']

for i, metric in enumerate(metrics):
    ax = axes[i]
    vals = [results[m][metric] for m in results]
    bars = ax.bar(results.keys(), vals, color=colors, edgecolor='white')
    ax.set_title(metric, fontsize=14, fontweight='bold')
    ax.set_xticklabels(results.keys(), rotation=20, ha='right', fontsize=9)
    
    # Add value labels on bars
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.2f}' if metric != 'R2' else f'{val:.4f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../Backend_API/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved model_comparison.png')

## 4. Select Best Model (Lowest RMSE)

In [ ]:
best_model_name = comparison_df.index[0]
best_model = trained_models[best_model_name]
print(f'Best model: {best_model_name}')
print(f'RMSE: ₹{results[best_model_name]["RMSE"]:,.2f}')
print(f'R²: {results[best_model_name]["R2"]}')
print(f'MAE: ₹{results[best_model_name]["MAE"]:,.2f}')

In [ ]:
# Actual vs Predicted scatter
y_pred_best = best_model.predict(X_test_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_best, alpha=0.5, c='#3498db', edgecolors='white')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Price (₹)', fontsize=12)
plt.ylabel('Predicted Price (₹)', fontsize=12)
plt.title(f'{best_model_name}: Actual vs Predicted Prices', fontsize=14, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../Backend_API/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved actual_vs_predicted.png')

## 5. Feature Importance (SHAP)
Identifies the top price drivers in the housing market.

In [ ]:
if best_model_name == 'XGBoost':
    explainer = shap.TreeExplainer(best_model)
elif best_model_name in ['Random Forest']:
    explainer = shap.TreeExplainer(best_model)
else:
    # For Linear Regression / MLPRegressor, use KernelExplainer on a sample
    X_sample = X_test_scaled[:50]
    explainer = shap.KernelExplainer(best_model.predict, X_sample)

shap_values = explainer.shap_values(X_test_scaled[:100])

feature_names = list(X.columns)

# SHAP summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_scaled[:100], feature_names=feature_names, show=False)
plt.title(f'SHAP Feature Importance — {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../Backend_API/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved feature_importance.png')

In [ ]:
# Print SHAP-based feature ranking
shap_abs_mean = np.abs(shap_values).mean(axis=0)
feature_ranking = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': shap_abs_mean
}).sort_values('Mean |SHAP|', ascending=False)

print('Top Price Drivers (SHAP):')
for i, row in feature_ranking.iterrows():
    print(f'  {i+1}. {row["Feature"]}: {row["Mean |SHAP|"]:.2f}')

feature_ranking

## 6. Save Artifacts

In [ ]:
# Save best model
with open('../Backend_API/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f'✅ Saved best model ({best_model_name}) → model.pkl')

# Save scaler
with open('../Backend_API/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Saved scaler → scaler.pkl')

# Save label encoders
with open('../Backend_API/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print('✅ Saved label encoders → label_encoders.pkl')

# Save feature columns
with open('../Backend_API/feature_columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)
print('✅ Saved feature columns → feature_columns.pkl')

# Save model metrics for the API
model_metrics = {
    'dataset_size': len(df),
    'feature_count': len(X.columns),
    'best_model': best_model_name,
    'comparison': results,
    'feature_importance': feature_ranking.to_dict('records')
}

with open('../Backend_API/model_metrics.pkl', 'wb') as f:
    pickle.dump(model_metrics, f)
print('✅ Saved model metrics → model_metrics.pkl')

print('\n🎯 All artifacts saved successfully!')

In [ ]:
# Final summary
print('=' * 60)
print('📊 SMART HOUSE PRICE PREDICTOR — TRAINING SUMMARY')
print('=' * 60)
print(f'Dataset: {len(df)} houses, {len(X.columns)} features')
print(f'Best Model: {best_model_name}')
print(f'R² Score: {results[best_model_name]["R2"]}')
print(f'RMSE: ₹{results[best_model_name]["RMSE"]:,.2f}')
print(f'MAE: ₹{results[best_model_name]["MAE"]:,.2f}')
print('\nModel Rankings:')
for i, (name, _) in enumerate(comparison_df.iterrows()):
    r = results[name]
    print(f'  {i+1}. {name}: R²={r["R2"]}, RMSE=₹{r["RMSE"]:,.2f}')
print('\nTop Price Drivers:')
for i, row in feature_ranking.iterrows():
    print(f'  {i+1}. {row["Feature"]}')
print('=' * 60)